# Trelis Tara zero-shot baseline on OpenSLR 104 test

This notebook runs [`Trelis/tara`](https://huggingface.co/Trelis/tara) zero-shot on the OpenSLR 104 Hinglish test set (2,942 clips, 4.76 h) and scores it with the project normalizer and metrics in `src/hinglish_asr/`. It runs in Google Colab on an L4 GPU.

Tara is Whisper large-v3 fine-tuned for Hindi and Hinglish. Its model card says it writes English words in Latin script and Hindi in Devanagari, the project's target format. It is the planned distillation teacher (model 7 in the README), so its score is the upper limit for the distilled student.

- **Inputs:** the project folder on Google Drive, and `openslr104_test.zip` (the clips in `data/processed/openslr104/test/`).
- **Decoding:** the code-switched prompt from the model card (`<|hi|><|mixedcode|>`), bf16, and the same temperature fallback as the whisper-small run (0.0-1.0, compression ratio 1.35, logprob -1.0, seed 0). Clips over 30 s (3 clips) are cut into 30 s pieces and the texts joined.
- **Outputs:** one row in `results/summary.csv`, and each clip's output in `results/tara-zeroshot/hyps.jsonl`.

Note: with the current scoring code T-WER equals WER, because `Normalizer.normalize` converts Devanagari loanwords to English before `t_wer` runs.

## Setup

In [1]:
from google.colab import drive

drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/projects/asr"
ZIP_PATH = f"{PROJECT}/openslr104_test.zip"
AUDIO_DIR = "/content/openslr104_test"

Mounted at /content/drive


In [2]:
!pip install -q jiwer

In [3]:
import os
import sys

import torch

os.chdir(PROJECT)
sys.path.insert(0, f"{PROJECT}/src")
print(torch.cuda.get_device_name(0))

NVIDIA L4


In [4]:
import zipfile

from tqdm.auto import tqdm

with zipfile.ZipFile(ZIP_PATH) as z:
    for member in tqdm(z.infolist(), desc="unzip"):
        z.extract(member, AUDIO_DIR)

unzip:   0%|          | 0/5886 [00:00<?, ?it/s]

## Test manifest

The manifest's `audio_path` values are local paths on the machine that made the data, so each clip is found by file name in the unzipped folder.

In [5]:
import json
from pathlib import Path

wavs = {p.name: str(p) for p in Path(AUDIO_DIR).rglob("*.wav")}
with open("data/processed/openslr104/test_manifest.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

missing = [r["utt_id"] for r in rows if Path(r["audio_path"]).name not in wavs]
assert not missing, f"{len(missing)} clips missing, e.g. {missing[:3]}"

paths = [wavs[Path(r["audio_path"]).name] for r in rows]
refs = [r["text"] for r in rows]
hours = sum(r["duration"] for r in rows) / 3600
print(f"{len(rows)} clips in manifest, {len(wavs)} wav files, {hours:.2f} h")
print(refs[0])
print(paths[0])

2942 clips in manifest, 5884 wav files, 4.76 h
libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
/content/openslr104_test/test/598753_406yMKxIdSDHRf8H_0000.wav


## Audio

Whisper reads at most 30 s at a time. Each clip becomes one or more `(clip index, audio)` pieces of up to 30 s, in order, so the texts of a long clip can be joined back after transcription.

In [6]:
import soundfile as sf

SR = 16000
CHUNK = 30 * SR

pieces = []
for i, path in enumerate(tqdm(paths, desc="read audio")):
    audio, sr = sf.read(path, dtype="float32")
    assert sr == SR and audio.ndim == 1, (path, sr, audio.shape)
    pieces += [(i, audio[s:s + CHUNK]) for s in range(0, len(audio), CHUNK)]

print(f"{len(pieces)} pieces from {len(paths)} clips")

read audio:   0%|          | 0/2942 [00:00<?, ?it/s]

2945 pieces from 2942 clips


## Model

The Hugging Face `pipeline` cannot take the custom `<|mixedcode|>` prompt, so the notebook calls `generate` directly. `decoder_input_ids` sets the full prompt. `language="hi"` only stops `generate` from running language detection, since the prompt already sets the language. The model's generation config caps the prompt plus the output at 448 tokens.

In [7]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor, set_seed

MODEL = "Trelis/tara"

processor = WhisperProcessor.from_pretrained(MODEL)
model = WhisperForConditionalGeneration.from_pretrained(MODEL, dtype=torch.bfloat16).to("cuda").eval()

prompt_tokens = ["<|startoftranscript|>", "<|hi|>", "<|mixedcode|>", "<|transcribe|>", "<|notimestamps|>"]
prompt_ids = processor.tokenizer.convert_tokens_to_ids(prompt_tokens)
assert processor.tokenizer.unk_token_id not in prompt_ids, dict(zip(prompt_tokens, prompt_ids))
PROMPT = torch.tensor([prompt_ids], device="cuda")

GENERATE = {
    "language": "hi",
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "compression_ratio_threshold": 1.35,
    "logprob_threshold": -1.0,
}


def transcribe(audios):
    features = processor.feature_extractor(audios, sampling_rate=SR, return_tensors="pt").input_features
    with torch.inference_mode():
        out = model.generate(
            features.to("cuda", torch.bfloat16),
            decoder_input_ids=PROMPT.repeat(len(audios), 1),
            **GENERATE,
        )
    return [t.strip() for t in processor.batch_decode(out, skip_special_tokens=True)]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

A check on 5 clips before the full run:

In [8]:
for (i, _), hyp in zip(pieces[:5], transcribe([a for _, a in pieces[:5]])):
    print("REF:", refs[i])
    print("HYP:", hyp, "\n")

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


REF: libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
HYP: LibreOffice Impress पर slide master और slide design इस spoken tutorial में आपका स्वागत है। 

REF: इस tutorial में हम यह सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें
HYP: इस tutorial में हम सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें। 

REF: background slide पर लागू किये गये सभी रंगों और प्रभावों को उद्घृत करता है जो विषयवस्तु के पीछे मौजूद हैं
HYP: background slide पर लागू किए गए सभी रंगों और प्रभावों को उद्घृत करता है जो विषय वस्तु के पीछे मौजूद है। 

REF: libre office impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते हैं
HYP: LibreOffice Impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद कर सकते हैं। 

REF: आप भी अपने खुद़ के custom backgrounds बना सकते हैं
HYP: आप भी अपने खुद के custom backgrounds बना सकते हैं। 



## Transcribe the test set

In [9]:
import time

BATCH = 16
set_seed(0)

texts = [[] for _ in rows]
start = time.time()
for b in tqdm(range(0, len(pieces), BATCH), desc="transcribe"):
    batch = pieces[b:b + BATCH]
    for (i, _), text in zip(batch, transcribe([a for _, a in batch])):
        texts[i].append(text)
elapsed = time.time() - start

hyps = [" ".join(t) for t in texts]
print(f"{len(hyps)} outputs, {sum(not h for h in hyps)} empty")
print(f"{elapsed / 60:.1f} min, real-time factor {elapsed / (hours * 3600):.4f}")

transcribe:   0%|          | 0/185 [00:00<?, ?it/s]

2942 outputs, 0 empty
34.5 min, real-time factor 0.1207


Each clip's output is saved before scoring, so the clips can be re-scored later without the GPU.

In [10]:
out_dir = Path("results/tara-zeroshot")
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "hyps.jsonl", "w", encoding="utf-8") as f:
    for r, h in zip(rows, hyps):
        f.write(json.dumps({"utt_id": r["utt_id"], "ref": r["text"], "hyp": h}, ensure_ascii=False) + "\n")

## Score

In [11]:
from hinglish_asr.metrics import score
from hinglish_asr.utils import Normalizer

norm = Normalizer("data/lists/loanwords.csv")
assert score(refs, refs, norm) == {"wer": 0.0, "t_wer": 0.0}

scores = score(refs, hyps, norm)
print(f"WER {scores['wer']:.4f}  T-WER {scores['t_wer']:.4f}")

WER 0.2950  T-WER 0.2950


Normalized text that the metrics compare, for the first 5 clips:

In [12]:
for ref, hyp in zip(refs[:5], hyps[:5]):
    print("REF:", norm.for_scoring(ref))
    print("HYP:", norm.for_scoring(hyp), "\n")

REF: libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
HYP: libreoffice impress पर slide master और slide design इस spoken tutorial में आपका स्वागत है 

REF: इस tutorial में हम यह सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें
HYP: इस tutorial में हम सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें 

REF: background slide पर लागू किये गये सभी रंगों और प्रभावों को उद्घृत करता है जो विषयवस्तु के पीछे मौजूद हैं
HYP: background slide पर लागू किए गए सभी रंगों और प्रभावों को उद्घृत करता है जो विषय वस्तु के पीछे मौजूद है 

REF: libre office impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद करते हैं
HYP: libreoffice impress में कई background options हैं जो बेहतर प्रस्तुतियाँ बनाने में आपकी मदद कर सकते हैं 

REF: आप भी अपने खुद़ के custom backgrounds बना सकते हैं
HYP: आप भी अपने खुद के custom backgrounds बना सकते हैं 



## Results file

In [13]:
import csv
import datetime

summary = Path("results/summary.csv")
row = {
    "model": MODEL,
    "dataset": "openslr104",
    "split": "test",
    "n_utts": len(rows),
    "wer": round(scores["wer"], 4),
    "t_wer": round(scores["t_wer"], 4),
    "decoding": "zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi + mixedcode, bf16, 30 s pieces",
    "date": datetime.date.today().isoformat(),
}
write_header = not summary.exists()
with open(summary, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(row)

print(summary.read_text(encoding="utf-8"))

model,dataset,split,n_utts,wer,t_wer,decoding,date
openai/whisper-small,openslr104,test,2942,0.915,0.915,"zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi, fp16, 30 s chunks",2026-11-23
Trelis/tara,openslr104,test,2942,0.295,0.295,"zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi + mixedcode, bf16, 30 s pieces",2026-11-23

